In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import keras.ops as K
from keras.layers import Input, Flatten, Dense
from keras.optimizers import Adam
from keras.metrics import BinaryAccuracy

# from keras.models import Sequential
from deel.lip.model import Sequential

from deel.lip.layers import (
    SpectralDense,
    SpectralConv2D,
    ScaledL2NormPooling2D,
    FrobeniusDense,
)
from deel.lip.activations import GroupSort, GroupSort2
from deel.lip.losses import HKR, KR, HingeMargin, MulticlassHKR, MulticlassKR

import numpy as np
import decomon

import sys

# setting path
sys.path.append('..')

from data_processing import load_data, select_data_for_radius_evaluation_MNIST08
from radius_evaluation_tools import compute_binary_certificate, starting_point_dichotomy
from lipschitz_decomon_tools import get_local_maximum, echantillonner_boule_l2_simple, echantillonner_boule_l2_simple_surbord

# Data Loading

In [2]:
x_train, x_test, y_train, y_test, y_test_ord = load_data("MNIST08")

In [3]:
model_path = "/home/aws_install/robustess_project/lip_models/demo3_FC_vanilla_MNIST08_channelfirst_False_disj_Neurons_single_output.keras"
model_bin = keras.models.load_model(model_path)
model_bin.compile(
   
    loss=HKR(
        alpha=10.0, min_margin=1.0
    ),  # HKR stands for the hinge regularized KR loss
    metrics=[
        # KR,  # shows the KR term of the loss
        HingeMargin(min_margin=1.0),  # shows the hinge term of the loss
    ],
    optimizer=Adam(learning_rate=0.001),)

model_bis = keras.models.load_model("/home/aws_install/robustess_project/lip_models/demo3_FC_vanilla_MNIST08_channelfirst_False_disj_Neurons_single_output_converted_4logits.keras")
model_bis.compile(
        # decreasing alpha and increasing min_margin improve robustness (at the cost of accuracy)
        # note also in the case of lipschitz networks, more robustness require more parameters.
        loss=MulticlassHKR(alpha=100, min_margin=0.25),
        optimizer=Adam(1e-4),
        metrics=["accuracy", MulticlassKR()],)

images, labels, idx_list = select_data_for_radius_evaluation_MNIST08(x_test, y_test_ord, model_bis)

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 12 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 14 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


# Selection of studied data

In [4]:
pt_choosen = 1
x = images[pt_choosen:pt_choosen+1].flatten().detach().cpu().numpy()
label = labels[pt_choosen:pt_choosen+1]
eps=5

In [5]:
yi = echantillonner_boule_l2_simple(x,eps)

# Dataset Creation

We want to learn an affine model that is close and overapproximate norm(x,y), hence, we create a dataset with generated samples

In [6]:
def norm(x,y):
    return np.linalg.norm(x-y)

In [7]:
def create_dataset_bord(nb, x, yi, eps):    
    input = []
    label = []
    for _ in range(nb):
        x_current = echantillonner_boule_l2_simple_surbord(x,eps)
        input.append(x_current)
        label.append(norm(x_current, yi))
    return np.array(input), np.array(label)

In [8]:
inputs, labels = create_dataset_bord(10000, x, yi, eps)

In [9]:
test_inputs, test_labels = create_dataset_bord(1000, x, yi, eps)

In [10]:
inputs.shape

(10000, 784)

# Model Training
We create a single layer affine model without activations and train it with a custom loss

In [11]:
# --- 1. Définition du Modèle Affine (Keras) ---
def create_affine_model(input_dim_model):
    model = keras.Sequential([Input(input_dim_model),
        keras.layers.Dense(1, activation=None, name="affine_layer")
    ], name="simple_affine_network")
    return model

In [12]:
def sur_approximation_mse_loss(y_true_norm, y_pred_affine):
    # y_pred_affine: Sortie du modèle (Wx + b_nn), shape [batch_size, 1]
    # y_true_norm: Norme L2 cible (||x - x0||), shape [batch_size,]
    
    y_pred_affine_squeezed = K.squeeze(y_pred_affine) # Shape [batch_size,]
    
    # Terme 1 (MSE): (g_i - f_i)^2, où g_i = y_pred_affine, f_i = y_true_norm
    mse_gap_term = K.square(y_pred_affine_squeezed - y_true_norm)
    
    # Terme 2 (Pénalité): lambda * ReLU(f_i - g_i)^2
    violation = y_true_norm - y_pred_affine_squeezed # Positif si g_i < f_i (violation)
    # penalty_term = LAMBDA_PENALTY_TF * K.relu(violation)
    penalty_term = 10 * K.square(K.relu(violation))
    
    # Perte moyenne sur le batch
    loss = K.mean(mse_gap_term + penalty_term)
    return loss

In [13]:
model = create_affine_model(x_train[0].flatten().shape)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=10e-3),
              loss=sur_approximation_mse_loss, metrics=['mse'])
            # loss='mse', metrics=['mse'])

In [14]:
model.fit(inputs, labels,
                    epochs=30,
                    # steps_per_epoch=50,
                    batch_size=128,
                    verbose=1) # verbose=1 pour la barre de progression

Epoch 1/30


79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 35.5715 - mse: 9.4902
Epoch 2/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2088 - mse: 0.1256
Epoch 3/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0912 - mse: 0.0634
Epoch 4/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0650 - mse: 0.0463
Epoch 5/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0559 - mse: 0.0385
Epoch 6/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0522 - mse: 0.0358
Epoch 7/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0510 - mse: 0.0350
Epoch 8/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0492 - mse: 0.0340
Epoch 9/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0478 - mse: 0.0329
Epoch 10/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0444 - mse: 0.0304
Epoch 11/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0439 - mse: 0.0308
Epoch 12/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0440 - mse: 0.0303
Epoch 13/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 

In [15]:
model.evaluate(test_inputs, test_labels)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0372 - mse: 0.0179


[0.04165539890527725, 0.018056346103549004]

In [16]:
count = 0
for i in range(test_inputs.shape[0]):
    x_current = test_inputs[i]
    if (norm(x_current, yi) - model(x_current[None]))<0:
        count += 1
result = count / test_inputs.shape[0] 
print(result)

0.732


# Calcul hyperplan surrapproximateur

In [17]:
def calculer_hyperplan_surestimateur(n_dim, fonction_convexe, centre_boule=None, rayon_boule=1.0):
    """
    Calcule les coefficients (a, b) d'un hyperplan z(x) = a^T x + b.
    L'hyperplan interpole fonction_convexe en n_dim + 1 points
    sur le bord d'une boule et la surestime (localement).

    Args:
        n_dim (int): Dimension de l'espace.
        fonction_convexe (callable): Fonction f(x) convexe.
        centre_boule (np.array, optional): Centre de la boule. Par défaut, origine.
        rayon_boule (float, optional): Rayon de la boule. Par défaut, 1.0.

    Returns:
        tuple: (a, b) où 'a' est le vecteur normal et 'b' est le terme constant.
               Retourne aussi les points d'interpolation et leurs valeurs f(x_i) pour vérification.
    """
    num_points_interpolation = n_dim + 1

    # 1. Générer des points sur le bord de la boule
    # Points aléatoires sur la sphère, puis translatés/mis à l'échelle
    directions_aleatoires = np.random.randn(num_points_interpolation, n_dim)
    normes = np.linalg.norm(directions_aleatoires, axis=1, keepdims=True)
    # Éviter la division par zéro (très peu probable avec randn)
    normes[normes == 0] = 1.0
    points_sur_sphere_unite = directions_aleatoires / normes
    points_sur_bord = centre_boule + rayon_boule * points_sur_sphere_unite

    # 2. Évaluer la fonction convexe aux points générés
    valeurs_y = np.array([fonction_convexe(p) for p in points_sur_bord])

    # 3. Construire et résoudre le système linéaire M * coeffs = valeurs_y
    #    où coeffs = [a_1, ..., a_n, b]^T
    #    Chaque ligne de M est [p_1, ..., p_n, 1]
    matrice_systeme = np.hstack([points_sur_bord, np.ones((num_points_interpolation, 1))])

    try:
        coefficients = np.linalg.solve(matrice_systeme, valeurs_y)
    except np.linalg.LinAlgError:
        # En cas de problème de singularité (peu probable avec des points aléatoires),
        # utiliser les moindres carrés comme solution de repli.
        print("Avertissement: np.linalg.solve a échoué, utilisation de np.linalg.lstsq.")
        coefficients = np.linalg.lstsq(matrice_systeme, valeurs_y, rcond=None)[0]

    a = coefficients[:-1]  # Les n_dim premiers coefficients
    b = coefficients[-1]   # Le dernier coefficient (terme constant)

    return a, b, points_sur_bord, valeurs_y

In [18]:
a, b, points_sur_bord, valeurs_y = calculer_hyperplan_surestimateur(784, lambda z : norm(z, yi), x, eps)

In [19]:
np.linalg.norm(points_sur_bord[9] - x)

5.0

In [20]:
i = 3

In [21]:
a@(inputs[i])+b

5.012804428835093

In [22]:
labels[i]

5.012629237260129

In [36]:
norm(inputs[i], yi)

4.996930462006564

In [35]:
model(inputs[i][None])

tensor([[5.0038]], device='cuda:0', grad_fn=<AddBackward0>)

is the hyperplane over the norm ?

In [37]:
i = 4
model(inputs[i][None]) - norm(inputs[i], yi)

tensor([[0.0806]], device='cuda:0', grad_fn=<SubBackward0>)

In [38]:
W, b = model.get_weights()

In [27]:
np.max(W)

0.0877957

In [28]:
b

array([0.1152635], dtype=float32)

In [29]:
W[:,0]@(inputs[i])+b

array([5.0831037], dtype=float32)

In [30]:
model(inputs[i][None])

tensor([[5.0831]], device='cuda:0', grad_fn=<AddBackward0>)

In [31]:
def function_to_optimize_all(x, label, W_list, b_list, y_list, model, L=1):
    # function we want to optimize, combination of lipschitz constraints in all yi
    outputs = []
    for i in range(len(y_list)):
        if label == 0:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] +\
                L*np.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)
            function = np.min(outputs)
        else:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] -\
                L*np.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)
            function = np.max(outputs)    
    return function

In [32]:
x_ball_center = x
x_ball_center = np.asarray(x_ball_center, dtype=np.float64)
y_list = []
for i in range(10):
    y_list.append(echantillonner_boule_l2_simple(x, eps))
# l = x_ball_center-eps
# u = x_ball_center+eps
l = x-eps
u = x+eps

W_list = []
b_list = []
for y_i in y_list:
    W, b, _, _ = calculer_hyperplan_surestimateur(784, lambda z : norm(z, yi), x, eps)
    W_list.append(W)
    b_list.append(b)

# Define the constraint: ||x - x_centre||_2**2 <= eps**2
def unit_ball_constraint(x, x_ball_center, eps):
        return eps**2 - np.linalg.norm(x - x_ball_center)**2

def jacobian_unit_ball_constraint(x, x_ball_center, eps):
        """
        Jacobien (gradient) de la fonction unit_ball_constraint.
        Retourne -x / ||x||_2.
        Non défini à x = 0.
        """
        # norm_x = np.linalg.norm(x)
        # return -x / norm_x
        return -2*(x - x_ball_center)

args_contrainte = (x_ball_center, eps)
    # Set up the constraint dictionary
constraints = ({
        'type': 'ineq',  # Inequality constraint: constraint(x) >= 0
        'fun': unit_ball_constraint,
        'jac': jacobian_unit_ball_constraint,
        'args': args_contrainte
})


In [33]:
from scipy.optimize import minimize

In [34]:
# Run the optimizer
if label == 0:
    result = minimize(fun=lambda x :-function_to_optimize_all(x, label, W_list, b_list, y_list, model, 1),\
    # jac= lambda x :-jac_function_to_optimize(x, label, W_list, b_list, y_list, model, L),\
    x0 = x_ball_center, method='SLSQP', constraints=constraints)
else:
    result = minimize(fun=lambda x :function_to_optimize_all(x, label, W_list, b_list, y_list, model, 1),\
    # jac= lambda x :jac_function_to_optimize(x, label, W_list, b_list, y_list, model, L),\
    x0 = x_ball_center, method='SLSQP', constraints=constraints)
# result = minimize(fun=lambda x :-function_to_optimize(x, W_1, b_1, y), x0 = x_ball_center, method='SLSQP', constraints=constraints)
# attention, le maximum est - result
# Display results
if result.success:
    if label == 0:
        print(result.x, -result.fun)
    else:
        print(result.x, result.fun)
else:
    print("Optimization failed:", result.message)
    raise ValueError(result.message)


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input tensor([[[[ 5.8599e-05, -3.5877e-02, -1.3615e-02, -1.0864e-01,  2.7473e-01,
            1.0805e-02, -2.0438e-01, -2.8550e-01,  9.8242e-02,  1.7795e-01,
           -2.0070e-01, -1.7899e-01, -1.7300e-01,  1.3521e-01,  1.8469e-01,
           -7.5277e-02, -5.4768e-02, -1.7366e-03, -2.2374e-01,  7.3663e-02,
           -2.3844e-01, -8.2265e-03, -9.6701e-02,  1.2519e-01, -9.0298e-02,
           -2.0953e-01,  1.3200e-01,  4.8057e-02],
          [-7.4471e-02, -6.6834e-02, -1.8915e-01, -7.2927e-02, -1.9669e-01,
           -6.5334e-02, -1.5735e-01, -1.2675e-01, -1.6974e-01,  1.9830e-01,
           -2.3093e-01, -1.3642e-01,  1.5307e-01, -5.4730e-03,  1.8303e-01,
           -2.3074e-01, -2.6110e-01,  1.6368e-01, -7.9792e-02,  4.6611e-02,
           -3.5958e-02, -8.7634e-02, -1.4094e-01, -1.5374e-01, -2.8341e-01,
            4.5160e-02,  1.7914e-01, -1.3337e-01],
          [-9.4556e-02,  7.9320e-02, 

# Empirical Tests

In [ ]:
x_current = echantillonner_boule_l2_simple(x,eps)

if (norm(x_current, yi) - model(x_current[None]))<0:
    print('the hyperplan is overapproximating : ', norm(x_current, yi) - model(x_current[None]))
else:
    print("error")



the hyperplan is overapproximating :  tensor([[-0.3889]], device='cuda:0', grad_fn=<RsubBackward1>)


# Optimisation

We know want to solve the optimisation problem, in order to know if our hyperplane is valid

In [ ]:
from scipy.optimize import minimize

In [ ]:
def function_to_optimize(x, x0, yi, W, b, eps):
    z0 = x0-yi
    return np.linalg.norm(z0) + eps + 2*z0@x - W@(z0 + x + yi) - b

In [ ]:
# Define the constraint: ||x - x_centre||_2**2 <= eps**2
def unit_ball_constraint(x, x_ball_center, eps):
    return eps - np.linalg.norm(x - x_ball_center)

def jacobian_unit_ball_constraint(x, x_ball_center, eps):
    return -2*(x - x_ball_center)

In [ ]:
x_ball_center = x
x_ball_center = np.asarray(x_ball_center, dtype=np.float64)

args_contrainte = (x_ball_center, eps)
# Set up the constraint dictionary
constraints = ({
    'type': 'eq',  # Inequality constraint: constraint(x) >= 0
    'fun': unit_ball_constraint,
    # 'jac': jacobian_unit_ball_constraint,
    'args': args_contrainte
})

In [ ]:
result = minimize(fun=lambda x :-function_to_optimize(x, x_ball_center, yi, W.squeeze(1), b, eps),\
        # jac= lambda x :-jac_function_to_optimize(x, label, W_list, b_list, y_list, model, L),\
        x0 = x_ball_center, method='SLSQP', constraints=constraints)

NameError: name 'W' is not defined

In [ ]:
if result.success:
    print(result.x, -result.fun)
else:
    print("Optimization failed:", result.message)   

[ 0.00603905  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905
  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905
  0.00603905 -0.05295046  0.00603905  0.00603905  0.00603905  0.00603905
  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905
  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905 -0.05295046
  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905
  0.00603905  0.00603905 -0.05295438  0.00603905  0.00603905  0.00603905
  0.00603905 -0.05295438 -0.05295438 -0.05295046  0.00603905  0.00603905
  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905
 -0.05295438  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905
  0.00603905  0.00603905  0.00603905 -0.05295046  0.00603905 -0.05295046
  0.00603905  0.00603905 -0.05295438 -0.05295438  0.00603905  0.00603905
 -0.05295438  0.00603905  0.00603905  0.00603905  0.00603905  0.00603905
  0.00603905  0.00603905  0.00603905  0.00603905  0

In [ ]:
-result.fun

1.488976001739502

In [ ]:
def optimize_hyperplane(x0, yi, W, b, eps):
    # Define the constraint: ||x - x_centre||_2**2 <= eps**2
    def unit_ball_constraint(x, x_ball_center, eps):
        return eps - np.linalg.norm(x - x_ball_center)

    
    x_ball_center = x0
    x_ball_center = np.asarray(x_ball_center, dtype=np.float64)

    args_contrainte = (x_ball_center, eps)
    # Set up the constraint dictionary
    constraints = ({
        'type': 'eq', 
        'fun': unit_ball_constraint,
        # 'jac': jacobian_unit_ball_constraint,
        'args': args_contrainte
    })
    result = minimize(fun=lambda x :-function_to_optimize(x, x_ball_center, yi, W.squeeze(1), b, eps),\
        # jac= lambda x :-jac_function_to_optimize(x, label, W_list, b_list, y_list, model, L),\
        x0 = x_ball_center, method='SLSQP', constraints=constraints)
    if result.success:
        return result.x, -result.fun
    else:
        print("Optimization failed:", result.message)
        return 0   

In [ ]:
_, fun = optimize_hyperplane(x, yi, W, b, eps)

In [ ]:
fun

1.488976001739502

In [ ]:
def f(z, W, b, y, label, model, L=1):
# Starting from W,b computed by training, we generate the lip bounding function.
    if label==0:
        # print(model(y.reshape((1,28,28))[None]).cpu().detach().numpy())
        return model(y.reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] +\
                L*(W@z +b) #scalar
    else:
        return model(y.reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] -\
                L*(W@z +b) #scalar


In [ ]:
def f_all(z, W_list, b_list, y_list, label, model, L=1):
# Compute the supremum of f (depending on the label) over all yi
    output = []
    for i in range(len(y_list)):
        output.append(f(z,W_list[i], b_list[i], y_list[i], label, model, L))
    if label==0:
        return np.max(output)
    else:
        return np.min(output)

In [ ]:
def generating_hyperplans(x_0, y_list, eps, optimization=False):
    W_list = []
    b_list = []
    for y_i in y_list:
        inputs, labels = create_dataset(2048, x_0, y_i, eps)
        model = create_affine_model(x_train[0].flatten().shape)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss=sur_approximation_mse_loss)
        model.fit(inputs, labels,
                    epochs=50,
                    steps_per_epoch=STEPS_PER_EPOCH,
                    verbose=0) # verbose=1 pour la barre de progression
        W, b = model.get_weights()
        
        if optimization:
            _, gap = optimize_hyperplane(x_0, y_i, W, b, eps)
            b = b + gap
            
        W_list.append(W.squeeze(1))
        b_list.append(b)
    return W_list, b_list

In [ ]:
y_list = []
for i in range(10):
    y_list.append(echantillonner_boule_l2_simple(x, eps))

In [ ]:
W_list, b_list = generating_hyperplans(x, y_list, eps)

In [ ]:
def function_to_optimize_all(z, label, x_0, y_list, eps, optimization, L):
    W_list, b_list = generating_hyperplans(x_0, y_list, eps, optimization)
    return f_all(z, W_list, b_list, y_list, label, L)

In [ ]:
def get_local_maximum(x, label, eps, y_list, model, L=1):
    # # Define your convex function
    # def f(x):
    #     # Example: quadratic function
    #     return np.dot(x, x) + 3 * x[0] - x[1]  # Replace with your actual function
    x_ball_center = x
    x_ball_center = np.asarray(x_ball_center, dtype=np.float64)

    W_list, b_list = generating_hyperplans(x, y_list, eps, True)

    # Define the constraint: ||x - x_centre||_2**2 <= eps**2
    def unit_ball_constraint(x, x_ball_center, eps):
        return eps**2 - np.linalg.norm(x - x_ball_center)**2

    def jacobian_unit_ball_constraint(x, x_ball_center, eps):
        """
        Jacobien (gradient) de la fonction unit_ball_constraint.
        Retourne -x / ||x||_2.
        Non défini à x = 0.
        """
        # norm_x = np.linalg.norm(x)
        # return -x / norm_x
        return -2*(x - x_ball_center)

    args_contrainte = (x_ball_center, eps)
    # Set up the constraint dictionary
    constraints = ({
        'type': 'ineq',  # Inequality constraint: constraint(x) >= 0
        'fun': unit_ball_constraint,
        'jac': jacobian_unit_ball_constraint,
        'args': args_contrainte
    })

    # Run the optimizer
    if label == 0:
        result = minimize(fun=lambda x :-f_all(x, W_list, b_list, y_list, label, model, L),\
        # jac= lambda x :-jac_function_to_optimize(x, label, W_list, b_list, y_list, model, L),\
        x0 = x_ball_center, method='SLSQP', constraints=constraints)
    else:
        result = minimize(fun=lambda x :f_all(x, W_list, b_list, y_list, label, model, L),\
        # jac= lambda x :jac_function_to_optimize(x, label, W_list, b_list, y_list, model, L),\
        x0 = x_ball_center, method='SLSQP', constraints=constraints)
    # result = minimize(fun=lambda x :-function_to_optimize(x, W_1, b_1, y), x0 = x_ball_center, method='SLSQP', constraints=constraints)
    # attention, le maximum est - result
    # Display results
    if result.success:
        if label == 0:
            return result.x, -result.fun
        else:
            return result.x, result.fun
    else:
        print("Optimization failed:", result.message)
        raise ValueError(result.message)

In [ ]:
pt_choosen = 0
eps = 0.9

In [ ]:
x = images[pt_choosen:pt_choosen+1].flatten().detach().cpu().numpy()

In [ ]:
y_list = []
for i in range(10):
    y_list.append(echantillonner_boule_l2_simple(x, eps))

In [ ]:
for i in range(10):
    y_list.append(echantillonner_boule_l2_simple(x, eps))

In [ ]:
_, fun = get_local_maximum(x, label, eps, y_list, model_bin)

Optimization failed: Iteration limit reached


TypeError: cannot unpack non-iterable int object

In [ ]:
fun

0.91268194

In [ ]:
fun

0.91268194

In [ ]:
fun

0.91268194

Can we retrieve this gap to b in order to get a better hyperplane ?